# Two-tower training on ESCI

This notebook trains the retrieval two-tower model on the prepared ESCI dataset (with optional query augmentation).

In [1]:
import sys
from pathlib import Path

# Project root
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"

## Prepare train/test splits (run once)

This cell makes sure `esci_train.parquet` and `esci_test.parquet` exist under `data/` by calling `load_esci`.

In [2]:
from src.data.load_data import load_esci
from src.data.utils import DATA_DIR as DEFAULT_DATA_DIR, EXAMPLES_FILENAME

# Decide where raw ESCI parquets live
if (DATA_DIR / EXAMPLES_FILENAME).exists():
    raw_data_dir = DATA_DIR
else:
    raw_data_dir = None  # use default ESCI_SUBDIR in load_esci

df = load_esci(data_dir=raw_data_dir, save_splits_dir=DATA_DIR)
print(f"Loaded {len(df)} rows; train/test splits written under {DATA_DIR}")

Loaded 601354 rows; train/test splits written under /Users/chen_bowen/AI & ML/Projects/Amazon_Search_Retrieval/data


## Configure training hyperparameters

Edit this dict to change learning rate, batch size, epochs, or query augmentation settings.

In [3]:
train_cfg = {
    "model_name": "all-MiniLM-L12-v2",  # slightly larger MiniLM
    "batch_size": 64,
    "epochs": 3,
    "lr": 2e-5,
    "temperature": 0.05,
    "use_query_augment": True,
    "augment_prob": 0.25,
    "save_path": DATA_DIR / "model.pt",
}
train_cfg

{'model_name': 'all-MiniLM-L12-v2',
 'batch_size': 64,
 'epochs': 3,
 'lr': 2e-05,
 'temperature': 0.05,
 'use_query_augment': True,
 'augment_prob': 0.25,
 'save_path': PosixPath('/Users/chen_bowen/AI & ML/Projects/Amazon_Search_Retrieval/data/model.pt')}

## Training helpers in this notebook

The next cell defines the small dataset wrapper, collate function, and training loop used for two-tower training (so you can see everything in one place).

In [4]:
import logging
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from src.data.query_augment import augment_query
from src.models.two_tower import TwoTowerEncoder
from tqdm.auto import tqdm


def contrastive_loss_with_reweighting(
    query_emb: torch.Tensor,  # [B, D] L2‑normalized query embeddings
    product_emb: torch.Tensor,  # [B, D] L2‑normalized product embeddings (positive on the diagonal)
    temperature: float = 0.05,
    reweight_hard: bool = True,
    hard_weight_power: float = 1.0,  # Hyper‑parameter β that controls how much we up‑weight hard negatives
) -> torch.Tensor:
    """Contrastive loss with hard-negative reweighting (HCL; Robinson et al., ICLR 2021).

    - If `reweight_hard` is False or `hard_weight_power <= 0`, this is standard InfoNCE.
    - Otherwise we approximate the debiased Hardness-aware Contrastive Loss (HCL):
      hard negatives (high sim) contribute more in the denominator.
    """
    B = query_emb.size(0)
    device = query_emb.device
    logits = torch.mm(query_emb, product_emb.t()) / temperature  # [B, B]
    labels = torch.arange(B, device=device, dtype=torch.long)

    if not reweight_hard or hard_weight_power <= 0:
        return F.cross_entropy(logits, labels)

    beta = hard_weight_power
    tau_pos = 0.1
    tau_neg = 1.0 - tau_pos

    Z_beta = torch.logsumexp(beta * logits, dim=1, keepdim=True) - torch.log(
        torch.tensor(B, dtype=torch.float, device=device)
    )
    Z_beta = torch.exp(Z_beta)  # [B, 1]

    diag_logits = torch.diag(logits)
    Z_pos_beta = torch.exp(beta * diag_logits).mean()  # scalar

    neg_exp = (Z_beta - tau_pos * Z_pos_beta) / tau_neg  # [B, 1]

    pos_logits = torch.diag(logits).unsqueeze(1)  # [B, 1]
    logits_eff = pos_logits + torch.log(B * neg_exp + 1e-8)

    return F.cross_entropy(logits_eff.squeeze(1), labels)


class QueryProductDataset(Dataset):
    """(query, product_text) pairs filtered to positive relevance, with optional query augmentation."""

    def __init__(
        self,
        df: pd.DataFrame,
        min_relevance: int = 2,
        use_query_augment: bool = True,
        augment_prob: float = 0.25,
    ) -> None:
        self.df = (
            df[df["relevance"] >= min_relevance][["query", "product_text"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
        self.use_query_augment = use_query_augment
        self.augment_prob = augment_prob

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[str, str]:
        row = self.df.iloc[idx]
        query = str(row["query"])
        if self.use_query_augment:
            query = augment_query(query, prob=self.augment_prob)
        return query, str(row["product_text"])


def collate_query_product(batch: list[tuple[str, str]]) -> tuple[list[str], list[str]]:
    queries, products = zip(*batch)
    return list(queries), list(products)


def train_two_tower(
    data_dir: Path,
    *,
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    epochs: int = 3,
    lr: float = 2e-5,
    temperature: float = 0.05,
    reweight_hard: bool = True,
    hard_weight_power: float = 1.0,
    use_query_augment: bool = True,
    augment_prob: float = 0.25,
    save_path: Path | None = None,
) -> TwoTowerEncoder:
    """End-to-end two-tower training on `esci_train.parquet` under data_dir."""
    train_path = Path(data_dir) / "esci_train.parquet"
    if not train_path.exists():
        raise FileNotFoundError(f"Train split not found at {train_path}. Run the split cell above first.")
    train_df = pd.read_parquet(train_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TwoTowerEncoder(model_name=model_name, shared=False, normalize=True).to(device)

    dataset = QueryProductDataset(
        train_df,
        use_query_augment=use_query_augment,
        augment_prob=augment_prob,
    )
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_query_product,
        num_workers=0,
        drop_last=True,
    )

    logging.info(
        "Training config: model=%s, batch_size=%d, epochs=%d, lr=%.2e, temperature=%.3f, "
        "reweight_hard=%s, hard_weight_power=%.2f, use_query_augment=%s, augment_prob=%.2f",
        model_name,
        batch_size,
        epochs,
        lr,
        temperature,
        reweight_hard,
        hard_weight_power,
        use_query_augment,
        augment_prob,
    )
    logging.info(
        "Train data: %d rows before filtering, %d positive (query, product_text) pairs after filtering",
        len(train_df),
        len(dataset),
    )
    logging.info("Using device: %s", device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        for queries, products in tqdm(
            dataloader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            leave=False,
        ):
            opt.zero_grad()
            q_emb, p_emb = model(query_strings=queries, product_strings=products, device=device)
            loss = contrastive_loss_with_reweighting(
                q_emb,
                p_emb,
                temperature=temperature,
                reweight_hard=reweight_hard,
                hard_weight_power=hard_weight_power,
            )
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_batches += 1
        logging.info("Epoch %s/%s loss=%.4f", epoch + 1, epochs, total_loss / max(n_batches, 1))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"model_state": model.state_dict(), "model_name": model_name}, save_path)
    return model

## Run training

This calls `train_two_tower` defined above so you can step into or modify the training loop directly.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

model = train_two_tower(
    data_dir=DATA_DIR,
    model_name=train_cfg["model_name"],
    batch_size=train_cfg["batch_size"],
    epochs=train_cfg["epochs"],
    lr=train_cfg["lr"],
    temperature=train_cfg["temperature"],
    use_query_augment=train_cfg["use_query_augment"],
    augment_prob=train_cfg["augment_prob"],
    save_path=train_cfg["save_path"],
)